In [ ]:
from tokenization.pae_preprocess import normalize_pae_text
from tokenization.pae_tokenizer import PAETokenizer

train_samples = [
    {"audio_path": "data/audio/0001.flac", "pae_text": "G-2   4/4   c d e|f g"},
    {"audio_path": "data/audio/0002.flac", "pae_text": "F-4\t3/4 a b c | d"},
]

normalized_train_texts = [
    normalize_pae_text(sample["pae_text"])
    for sample in train_samples
]

tokenizer = PAETokenizer.from_texts(normalized_train_texts)

print("Vocab size:", tokenizer.vocab.vocab_size)
print("Blank id:", tokenizer.vocab.blank_id)

In [ ]:
from torch.utils.data import DataLoader

from tokenization.pae_preprocess import normalize_pae_text
from tokenization.pae_tokenizer import PAETokenizer
from dataset_pipeline.audio_features import LogSTFTExtractor
from dataset_pipeline.dataset import AudioToScoreDataset
from dataset_pipeline.collate import ctc_collate_fn

train_samples = [
    {"audio_path": "data/audio/0001.flac", "pae_text": "G-2   4/4   c d e|f g"},
    {"audio_path": "data/audio/0002.flac", "pae_text": "F-4\t3/4 a b c | d"},
]

# Build vocab from normalized training texts
normalized_train_texts = [
    normalize_pae_text(sample["pae_text"])
    for sample in train_samples
]
tokenizer = PAETokenizer.from_texts(normalized_train_texts)

# Feature extractor
feature_extractor = LogSTFTExtractor(
    sample_rate=22050,
    n_fft=2048,
    hop_length=512,
    bins_per_octave=24,
    n_octaves=6,
)

# Dataset
dataset = AudioToScoreDataset(
    samples=train_samples,
    tokenizer=tokenizer,
    feature_extractor=feature_extractor,
    target_sample_rate=22050,
)

# DataLoader
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=ctc_collate_fn,
)

batch = next(iter(loader))

print("features:", batch["features"].shape)         # [B, T_max, F]
print("input_lengths:", batch["input_lengths"])     # [B]
print("targets:", batch["targets"].shape)           # [sum(U_i)]
print("target_lengths:", batch["target_lengths"])   # [B]
print("normalized pae_texts:", batch["pae_texts"])

In [ ]:
import torch

from CRNN_model.model import CRNN

B = 2
T = 300
F = 144
vocab_size = 30

features = torch.randn(B, T, F)
input_lengths = torch.tensor([300, 260], dtype=torch.long)

model = CRNN(
    input_freq_dim=F,
    vocab_size=vocab_size,
    conv_channels=(16, 16),
    lstm_hidden_size=512,
    lstm_num_layers=2,
)

output = model(features, input_lengths)

print("log_probs:", output.log_probs.shape)         # [T_out, B, vocab_size]
print("output_lengths:", output.output_lengths)     # [B]

In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

ctc_loss = nn.CTCLoss(
    blank=tokenizer.vocab.blank_id,
    reduction="mean",
    zero_infinity=True,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
from training.train import train_one_epoch, validate_one_epoch

# assuming you already created:
# train_dataset, val_dataset, tokenizer

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=ctc_collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=ctc_collate_fn,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CRNN(
    input_freq_dim=144,
    vocab_size=tokenizer.vocab.vocab_size,
    conv_channels=(16, 16),
    lstm_hidden_size=512,
    lstm_num_layers=2,
).to(device)

ctc_loss = torch.nn.CTCLoss(
    blank=tokenizer.vocab.blank_id,
    reduction="mean",
    zero_infinity=True,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        ctc_loss=ctc_loss,
        optimizer=optimizer,
        device=device,
        log_every=10,
    )

    val_loss = validate_one_epoch(
        model=model,
        loader=val_loader,
        ctc_loss=ctc_loss,
        device=device,
    )

    print(f"epoch={epoch} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

In [ ]:
from __future__ import annotations

from typing import Dict, List, Tuple

from decoding.decode import greedy_decode_batch
from decoding.metrics import corpus_cer, corpus_wer


@torch.no_grad()
def evaluate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    ctc_loss: nn.CTCLoss,
    tokenizer,
    device: torch.device,
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total_batches = 0

    all_refs: List[str] = []
    all_hyps: List[str] = []

    for batch in loader:
        batch = {
            "features": batch["features"].to(device),
            "input_lengths": batch["input_lengths"].to(device),
            "targets": batch["targets"].to(device),
            "target_lengths": batch["target_lengths"].to(device),
            "pae_texts": batch["pae_texts"],
            "audio_paths": batch["audio_paths"],
        }

        output = model(
            features=batch["features"],
            input_lengths=batch["input_lengths"],
        )

        loss = ctc_loss(
            output.log_probs,
            batch["targets"],
            output.output_lengths,
            batch["target_lengths"],
        )

        hyps = greedy_decode_batch(
            log_probs=output.log_probs,
            output_lengths=output.output_lengths,
            tokenizer=tokenizer,
        )

        refs = batch["pae_texts"]

        all_refs.extend(refs)
        all_hyps.extend(hyps)

        total_loss += float(loss.item())
        total_batches += 1

    avg_loss = total_loss / total_batches if total_batches > 0 else 0.0
    cer = corpus_cer(all_refs, all_hyps)
    wer = corpus_wer(all_refs, all_hyps)

    return {
        "loss": avg_loss,
        "cer": cer,
        "wer": wer,
    }

In [ ]:
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        ctc_loss=ctc_loss,
        optimizer=optimizer,
        device=device,
        log_every=10,
    )

    val_metrics = evaluate_one_epoch(
        model=model,
        loader=val_loader,
        ctc_loss=ctc_loss,
        tokenizer=tokenizer,
        device=device,
    )

    print(
        f"epoch={epoch} "
        f"train_loss={train_loss:.4f} "
        f"val_loss={val_metrics['loss']:.4f} "
        f"val_cer={val_metrics['cer']:.4f} "
        f"val_wer={val_metrics['wer']:.4f}"
    )

In [ ]:
@torch.no_grad()
def show_predictions(
    model,
    loader,
    tokenizer,
    device,
    num_examples: int = 5,
):
    model.eval()

    shown = 0

    for batch in loader:
        features = batch["features"].to(device)
        input_lengths = batch["input_lengths"].to(device)

        output = model(features, input_lengths)

        hyps = greedy_decode_batch(
            log_probs=output.log_probs,
            output_lengths=output.output_lengths,
            tokenizer=tokenizer,
        )

        refs = batch["pae_texts"]

        for ref, hyp in zip(refs, hyps):
            print("REF:", ref)
            print("HYP:", hyp)
            print("-" * 60)

            shown += 1
            if shown >= num_examples:
                return

In [ ]:
from decoding.pyctcdecode_utils import build_pae_ctc_decoder

pyctc_decoder = build_pae_ctc_decoder(tokenizer)

val_metrics = evaluate_one_epoch(
    model=model,
    loader=val_loader,
    ctc_loss=ctc_loss,
    tokenizer=tokenizer,
    device=device,
    decoding="pyctcdecode",
    pyctc_decoder=pyctc_decoder,
)

print(
    f"val_loss={val_metrics['loss']:.4f} "
    f"val_cer={val_metrics['cer']:.4f} "
    f"val_wer={val_metrics['wer']:.4f}"
)